In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd


Mounted at /content/drive


In [ ]:
!pip install transformers  peft accelerate faiss-cpu pypdf langchain langchain-community docarray datasets
from huggingface_hub import login
login(token="")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd

def contar_filas_excel(ruta_archivo, nombre_hoja=None):
    """
    Cuenta el número de filas con datos en un archivo Excel.

    Args:
        ruta_archivo (str): Ruta del archivo Excel.
        nombre_hoja (str): Nombre de la hoja específica a contar. Si es None, cuenta la primera hoja.

    Returns:
        int: Número de filas con datos.
    """
    try:
        if nombre_hoja:
            df = pd.read_excel(ruta_archivo, sheet_name=nombre_hoja)
        else:
            df = pd.read_excel(ruta_archivo)  # Lee la primera hoja por defecto

        # Elimina filas completamente vacías si las hubiera
        df = df.dropna(how='all')

        return len(df)
    except Exception as e:
        print(f"Error al procesar el archivo: {e}")
        return 0

# if __name__ == "__main__":
#     # Ejemplo de uso
#     archivo = input("Ingrese la ruta del archivo Excel: ").strip()

#     # Opción para seleccionar hoja específica
#     todas_hojas = input("¿Desea contar filas en todas las hojas? (s/n): ").strip().lower() == 's'

#     if todas_hojas:
#         # Contar filas en todas las hojas
#         try:
#             hojas = pd.ExcelFile(archivo).sheet_names
#             total_filas = 0

#             for hoja in hojas:
#                 filas = contar_filas_excel(archivo, hoja)
#                 print(f"Hoja '{hoja}': {filas} filas")
#                 total_filas += filas

#             print(f"\nTotal filas en todas las hojas: {total_filas}")
#         except Exception as e:
#             print(f"Error al leer el archivo: {e}")
#     else:
#         # Contar solo en la primera hoja o una específica
#         hoja_especifica = input("Ingrese el nombre de la hoja (deje vacío para la primera): ").strip()
#         if not hoja_especifica:
#             filas = contar_filas_excel(archivo)
#             print(f"\nTotal filas en la primera hoja: {filas}")
#         else:
#             filas = contar_filas_excel(archivo, hoja_especifica)
#             print(f"\nTotal filas en la hoja '{hoja_especifica}': {filas}")

Ingrese la ruta del archivo Excel: /content/drive/My Drive/Univalle/tesis/Tesis/resources/listado-informes.xlsx
¿Desea contar filas en todas las hojas? (s/n): s
Hoja 'Informes y Casos_Publicos': 715 filas

Total filas en todas las hojas: 715


In [ ]:
from pathlib import Path
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader, CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
import os
from pathlib import Path

# Ruta en tu Google Drive
PDF_FOLDER = "/content/drive/My Drive/Univalle/tesis/Tesis/reports-pdf"
RAGS_FOLDER = "/content/drive/My Drive/Univalle/tesis/RAGs/reports_faiss_index"




ModuleNotFoundError: Module langchain_community.embeddings not found. Please install langchain-community to access this module. You can install it using `pip install -U langchain-community`

In [ ]:
import pandas as pd

LISTING_PATH = "/content/drive/My Drive/Univalle/tesis/Tesis/resources/listado-informes.xlsx"
listing_df = pd.read_excel(LISTING_PATH)

# 2. Build FAISS index over title + description
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
listing_docs = []
for _, row in listing_df.iterrows():
    text = f"{row['ident']} {row['title']} {row['description']}"
    metadata = {"ident": str(row['ident']), "title": row['title'], "description": row['description']}
    listing_docs.append(Document(page_content=text, metadata=metadata))
faiss_reports = FAISS.from_documents(listing_docs, embeddings)

<ipython-input-4-2f025c0253d8>:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to acces

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
RAGS_FOLDER = "/content/drive/My Drive/Univalle/tesis/RAGs/reports_faiss_index"
faiss_reports.save_local(RAGS_FOLDER)

In [ ]:
#load fromlocal

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
faiss_reports = FAISS.load_local(RAGS_FOLDER, embeddings,allow_dangerous_deserialization=True)

In [ ]:
def get_relevant_reports(question: str, k: int = 3):
    # returns list of metadata dicts for top-k reports
    hits = faiss_reports.similarity_search_with_score(question, k=k)
    print("reportes encontrados: ",len(hits))
    return [hit[0].metadata for hit in hits]

# 4. Given a report ident, load its file and retrieve best context chunks

def retrieve_report_context(ident: str, question: str, k_chunks: int = 5):
    # Determine file path
    base_folder = Path(PDF_FOLDER)
    pdf_path = base_folder / f"{ident}.pdf"
    csv_path = base_folder / f"{ident}.csv"
    if pdf_path.exists():
        loader = PyPDFLoader(str(pdf_path))
    elif csv_path.exists():
        loader = CSVLoader(str(csv_path))
    else:
        print(f"No file found for ident {ident}")
        return []

    # Load and split pages
    pages = loader.load_and_split()

    # Build in-memory vectorstore
    vectordb = DocArrayInMemorySearch.from_documents(pages, embeddings)

    # Retrieve top-k context chunks
    chunks = vectordb.similarity_search(question, k=k_chunks)
    return [chunk.page_content for chunk in chunks]

# 5. Full pipeline: from question to context texts

def retrieve_contexts(question: str, top_reports: int = 5, top_chunks: int = 5):
    reports = get_relevant_reports(question, k=top_reports)
    all_contexts = []
    for rep in reports:
        contexts = retrieve_report_context(rep['ident'], question, k_chunks=top_chunks)
        all_contexts.extend(contexts)
    return all_contexts


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("mistralai/Ministral-8B-Instruct-2410", device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, "raulgdp/Mistral-8B-Instruct-2410-009", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("mistralai/Ministral-8B-Instruct-2410")



config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [ ]:
def answer_question(question, k_doc=3, max_tokens=150):
    similar_contexts = retrieve_contexts(question)
    context = "\n".join(similar_contexts)

    prompt = f"""
A continuación, se presenta una pregunta sobre el conflicto armado colombiano, junto con un contexto que proporciona información relevante. Escribe una respuesta que complete adecuadamente la solicitud.

Pregunta:
{question}

Contexto:
{context}

Respuesta:
"""
    # print("contexto",context)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Respuesta:")[-1].strip()
    return response


In [ ]:
question = "cuantas victimas han habido del conflicto?"
answer = answer_question(question)
print("Answer:", answer)

reportes encontrados:  5


ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advance

KeyboardInterrupt: 